In [1]:
from pathlib import Path
import sys, os, platform

CWD  = Path.cwd().resolve()
ROOT = CWD if (CWD / "src").exists() else CWD.parent
if str(ROOT) not in sys.path: sys.path.append(str(ROOT))

In [2]:
DOC_ID = "NFS_2019"   # e.g., "title17"
PDF = ROOT / "data" / "raw" / f"{DOC_ID}.pdf"
RUN_DIR = ROOT / "data" / "runs" / DOC_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
MD_DIR     = RUN_DIR / "md"
CHUNKS_DIR = RUN_DIR / "chunks"
INDEX_DIR  = RUN_DIR / "index"
SFT_DIR    = RUN_DIR / "sft"
PAIRS_DIR  = RUN_DIR / "pairs"

print(f"[run] DOC_ID={DOC_ID}")
print(f"[pdf] {PDF}")
print(f"[out] {RUN_DIR}")

[run] DOC_ID=NFS_2019
[pdf] D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\raw\NFS_2019.pdf
[out] D:\IIT BBS\Job Resources\Business Optima\pdf-agent\data\runs\NFS_2019


In [4]:
import io, fitz  # PyMuPDF
from PIL import Image
import pytesseract

PDF = ROOT / "data" / "raw" / "NFS_2019.pdf"   # adjust if needed

pages_to_test = [0, 9, 49]  # (1,10,50) 0-indexed
doc = fitz.open(str(PDF))

for i in pages_to_test:
    page = doc.load_page(i)
    pix = page.get_pixmap(dpi=300)   # 300 dpi is a good OCR baseline
    img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("L")

    # Reasonable OCR config for dense text
    text = pytesseract.image_to_string(img, lang="eng", config="--oem 1 --psm 6")
    print(f"\n[Page {i+1}] chars={len(text)}")
    print(text[:600].strip() or "(no text)")



[Page 1] chars=135
OFFICIAL
NEW YORK STATE WORKERS’ COMPENSATION
MEDICAL
FEE SCHEDULE
| Effective 4/1/2019 |
al Workers’
creortuniy. | Compensation
Board

[Page 10] chars=3062
Introduction and General Guidelines New York State Workers’ Compensation Medical Fee Schedule
“Rating Description ~~~ ~~+~*| ~~ ~+Rating + ‘Description = |.
“IM-NEPH Nephrology —~—~SCS*<“‘*YSC*S*éS =RWE'S”~S~*~*~*«Rematlogy———O—=~—~—‘—S
~IM-ONCL Medical Oncology == = === ~~=~*| +=P-SM ~—~—~—~—«sSportsMedicine—(‘éCS#
"IM-PD —__—*PulmonaryDiseases =0s=<“<i«~‘';;SPAS~*~*«éPatOdtlowYSC—“‘C™SOCOC#SNC*C#C*i‘(C(®’SS®S(N(NN(N(NCN
~IM-RHE —Rheumatology = ~=~*| ~+=PA-AP.~—SCAatomic Pathology =———™S
"IM-SM ___‘SportsMedicine =—=—S—=—<“—s~‘“‘;*LS*‘iA@B@];CSO*«*é@id Banking ss
‘NS ——~—SNeurological Surgery

[Page 50] chars=8
(
' : )


In [5]:
from src.ingest.pdf_to_markdown import convert_pdf_to_markdown
paths = convert_pdf_to_markdown(PDF, MD_DIR, ocr=True)
print("markdown pages:", paths)

markdown pages: {'markdown': WindowsPath('D:/IIT BBS/Job Resources/Business Optima/pdf-agent/data/runs/NFS_2019/md/NFS_2019.md'), 'toc_json': WindowsPath('D:/IIT BBS/Job Resources/Business Optima/pdf-agent/data/runs/NFS_2019/md/NFS_2019.toc.json'), 'pages_jsonl': WindowsPath('D:/IIT BBS/Job Resources/Business Optima/pdf-agent/data/runs/NFS_2019/md/NFS_2019.pages.jsonl'), 'images_dir': None}


In [6]:
import itertools, json
for line in itertools.islice(open(paths["pages_jsonl"], "r", encoding="utf-8"), 2):
    print(json.loads(line))

{'page': 1, 'text': 'OFFICIAL\nNEW YORK STATE WORKERS’ COMPENSATION\n\nMEDICAL\n\nFEE SCHEDULE\n\nEffective 4/1/2019\n\nNEWYORK | Workers.\nOPPORTUNITY. Compensation\nBoard\n', 'ocr_used': True}
{'page': 2, 'text': 'COPYRIGHT\n© 2018 State of New York\n\nFee data © 2018 Oputm360, LLC.\n\nCPT codes, descriptions, and two-digit numeric modifiers only, © 2017 American Medical\nAssociation\n\nAnesthesia base units only, © 2017 American Society of Anesthesiologists\n\nAll rights reserved. Printed in the United States of America. No part of this publication may be\nreproduced or transmitted in any form or by any means, electronic or mechanical, including\nphotocopy, recording, or storage in a database retrieval system, without the prior written\npermission of the publisher.\n\nMade in the USA\n\nOPTUM360”\n\n1.800.464.3649\n', 'ocr_used': True}


In [7]:
from src.ingest.md_to_chunks import md_to_chunks
CHUNKS = CHUNKS_DIR / "chunks.jsonl"
n = md_to_chunks(
    MD_DIR / f"{DOC_ID}.md", CHUNKS,
    pages_jsonl=MD_DIR / f"{DOC_ID}.pages.jsonl",
    max_chars=1600, overlap=400, drop_gibberish=True, drop_toc=False
)
print("chunks:", n)

chunks: 782
